# CIFAR-10: KNN vs CNN (PyTorch)

This notebook demonstrates:

1. Loading the CIFAR-10 dataset
2. Creating a simple baseline using KNN (flattened images)
3. Building a simple CNN with PyTorch
4. Comparing performance metrics

---


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

## 1. Load CIFAR-10 Dataset

In [3]:
transform = transforms.Compose([
    transforms.ToTensor()
])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                             download=True, transform=transform)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                            download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

classes = train_dataset.classes
classes

100.0%


Extracting ./data/cifar-10-python.tar.gz to ./data
Files already downloaded and verified


['airplane',
 'automobile',
 'bird',
 'cat',
 'deer',
 'dog',
 'frog',
 'horse',
 'ship',
 'truck']

## 2. Baseline Model: KNN (Flattened Images)

In [4]:
# For speed in class, use a subset
subset_size = 5000
X_train = train_dataset.data[:subset_size].reshape(subset_size, -1) / 255.0
y_train = np.array(train_dataset.targets[:subset_size])

X_test = test_dataset.data[:1000].reshape(1000, -1) / 255.0
y_test = np.array(test_dataset.targets[:1000])

knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, y_train)

y_pred_knn = knn.predict(X_test)

knn_acc = accuracy_score(y_test, y_pred_knn)
print("KNN Accuracy:", knn_acc)


/Users/eugenio/Documents/Computer_Vision/.venv/lib/python3.11/site-packages/threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


KNN Accuracy: 0.261


## 3. Simple CNN with PyTorch

In [5]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1) # Input channels = 3 (RGB), output channels = 16, kernel size = 3 
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1) # Input channels = 16, output channels = 32, kernel size = 3
        self.pool = nn.MaxPool2d(2, 2) # Kernel size = 2, stride = 2
        self.fc1 = nn.Linear(32 * 8 * 8, 128) # Input features = 32*8*8, output features = 128
        self.fc2 = nn.Linear(128, 10) # Input features = 128, output features = 10 (number of classes)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


## 4. Train the CNN

In [6]:
epochs = 5

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader):.4f}")


Epoch 1/5, Loss: 1.5703
Epoch 2/5, Loss: 1.2310
Epoch 3/5, Loss: 1.1056
Epoch 4/5, Loss: 1.0194
Epoch 5/5, Loss: 0.9529


## 5. Evaluate CNN

In [7]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

cnn_acc = accuracy_score(all_labels, all_preds)
print("CNN Accuracy:", cnn_acc)


CNN Accuracy: 0.6421


## 6. Comparison Summary

In [8]:
print("KNN Accuracy:", knn_acc)
print("CNN Accuracy:", cnn_acc)

KNN Accuracy: 0.261
CNN Accuracy: 0.6421
